# State-wise infant mortality and public health expenditure
## India: 2023 outcome data and 2024–25 revised expenditure estimates

This notebook combines RBI's 2023 state-wise infant mortality rate (IMR) with RBI's 2024–25 Revised Estimate (RE) for health expenditure as a share of aggregate expenditure. The reference periods differ. The analysis is exploratory and must not be interpreted causally.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../Dataset')
imr = pd.read_csv(DATA / 'state_infant_mortality_rate_2023.csv')
spend = pd.read_csv(DATA / 'state_health_expenditure_share_rbi_2024_25_re.csv')
imr.head(), spend.head()

## 1. Validate source coverage

Keep the all-India IMR aggregate out of state-level comparisons. The DNH observation includes Daman & Diu in the source and must not be double-counted.

In [ ]:
imr_states = imr[~imr['state_ut'].eq('All India')].copy()
print('IMR rows:', len(imr_states))
print('Missing IMR:', imr_states['infant_mortality_rate_2023_per_1000_live_births'].isna().sum())
print('Spending rows:', len(spend))
assert spend['state_ut'].is_unique
assert imr_states['state_ut'].is_unique

## 2. State-wise infant mortality rate, 2023

Lower IMR indicates fewer infant deaths per 1,000 live births. The measure reflects multiple determinants, not only health-system performance.

In [ ]:
valid_imr = imr_states.dropna(subset=['infant_mortality_rate_2023_per_1000_live_births']).sort_values('infant_mortality_rate_2023_per_1000_live_births')
plt.figure(figsize=(10, 11))
plt.barh(valid_imr['state_ut'], valid_imr['infant_mortality_rate_2023_per_1000_live_births'])
plt.xlabel('Infant deaths per 1,000 live births')
plt.ylabel('State / UT')
plt.title('State-wise infant mortality rate, 2023')
plt.tight_layout()
plt.show()

## 3. Exploratory comparison with 2024–25 health expenditure shares

The source periods are not identical: IMR is for calendar 2023, while spending is a 2024–25 Revised Estimate. This scatter plot is descriptive only. State/UT names are harmonised explicitly; the combined DNH/Daman & Diu observation is excluded from the matched analysis because it cannot be split into comparable independent units.

In [ ]:
imr_map = {'NCT Delhi':'Delhi', 'Jammu and Kashmir':'Jammu & Kashmir', 'Dadra & Nagar Haveli and Daman & Diu':'Dadra & Nagar Haveli (includes Daman & Diu)'}
imr_states['join_state'] = imr_states['state_ut'].replace(imr_map)
spend['join_state'] = spend['state_ut'].replace({'NCT Delhi':'Delhi', 'Dadra & Nagar Haveli & Daman & Diu':'Dadra & Nagar Haveli (includes Daman & Diu)'})
merged = spend.merge(imr_states[['join_state','infant_mortality_rate_2023_per_1000_live_births']], on='join_state', how='inner')
merged = merged.dropna(subset=['infant_mortality_rate_2023_per_1000_live_births'])
merged = merged[~merged['join_state'].eq('Daman & Diu')]
print('Matched observations:', len(merged))
plt.figure(figsize=(9, 6))
plt.scatter(merged['health_expenditure_share_aggregate_2024_25_re_pct'], merged['infant_mortality_rate_2023_per_1000_live_births'])
for _, row in merged.iterrows():
    plt.annotate(row['state_ut'], (row['health_expenditure_share_aggregate_2024_25_re_pct'], row['infant_mortality_rate_2023_per_1000_live_births']), fontsize=7, alpha=0.8)
plt.xlabel('Health expenditure (% of aggregate expenditure), 2024–25 RE')
plt.ylabel('Infant mortality rate, 2023 (per 1,000 live births)')
plt.title('Exploratory comparison: health budget share and IMR')
plt.tight_layout()
plt.show()
merged[['state_ut','health_expenditure_share_aggregate_2024_25_re_pct','infant_mortality_rate_2023_per_1000_live_births']].sort_values('infant_mortality_rate_2023_per_1000_live_births')

## 4. Interpretation and limitations

- The latest state outcome indicator used here is IMR for 2023; it is not a 2024–25 observation.
- The spending indicator is a 2024–25 Revised Estimate, not audited actual expenditure.
- Spending as a share of aggregate expenditure is not per-capita spending or spending as a percentage of GSDP.
- A cross-sectional association does not identify a causal effect. Income, maternal education, nutrition, sanitation, urbanisation, fertility and state demographics may confound the relationship.
- The combined Dadra & Nagar Haveli/Daman & Diu IMR observation is not separable in the source and should be treated cautiously.

## Sources

1. RBI, *Handbook of Statistics on Indian States 2024–25*, Table 4 (IMR): https://www.rbi.org.in/scripts/PublicationsView.aspx?id=23453
2. RBI, *State Finances: A Study of Budgets of 2025–26*, Statement 27 (health expenditure share): https://www.rbi.org.in/scripts/PublicationsView.aspx?id=23736